# Solutions — Sets: Deduplication, Membership, Set Algebra

Exercise notebook: `02_collections/03_sets.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Deduplicate while preserving order

**Working solution**

In [ ]:
def unique_in_order(items):
    return list(dict.fromkeys(items))

**Explanation**

Dicts preserve insertion order (guaranteed since Python 3.7), so `dict.fromkeys` is an ordered set. `list(set(items))` loses order.

**Explicit version (needed when the dedupe key differs from the item)**

In [ ]:
def unique_in_order_explicit(items):
    seen, result = set(), []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

**Complexity:** O(n).

**Common mistakes**

- `list(set(items))` — order is arbitrary.
- `if item not in result` on a list — O(n²).

## Exercise 2 — 🟡 Medium · Find orphaned foreign keys

**Working solution**

In [ ]:
def find_orphans(orders, customers):
    known = {c["customer_id"] for c in customers}
    referenced = {o.get("customer_id") for o in orders} - {None, ""}
    return referenced - known

**Explanation**

Two set comprehensions and a difference — the same thing as `LEFT JOIN ... WHERE c.id IS NULL`, in memory.

**Complexity:** O(n + m).

**Common mistakes**

- Checking `if cid not in [c['customer_id'] for c in customers]` inside a loop — rebuilds a list every iteration.

## Exercise 3 — 🟡 Medium · Schema drift detection

**Working solution**

In [ ]:
def _norm(col: str) -> str:
    return "_".join(col.strip().lower().split())

def compare_columns(expected, actual):
    exp = {_norm(c) for c in expected}
    normalized = [_norm(c) for c in actual]
    act = set(normalized)
    seen, duplicated = set(), set()
    for col in normalized:
        (duplicated if col in seen else seen).add(col)
    return {"missing": sorted(exp - act), "unexpected": sorted(act - exp),
            "common": sorted(exp & act), "duplicated": sorted(duplicated)}

**Explanation**

Set difference both ways plus intersection gives the full drift report. `(a if cond else b).add(x)` picks which set to mutate — cute, use sparingly.

**Complexity:** O(n log n) for sorting.

**Common mistakes**

- Comparing raw names so `'Order ID'` ≠ `'order_id'`.

## Exercise 4 — 🟠 Hard · Effective permissions with wildcards

**Working solution**

In [ ]:
def _expand(perms, universe):
    concrete = set()
    for perm in perms:
        if perm.endswith(":*"):
            prefix = perm[:-1]
            concrete |= {p for p in universe if p.startswith(prefix)}
        else:
            concrete.add(perm)
    return concrete

def effective_permissions(roles, role_perms, denied):
    universe = {p for perms in role_perms.values() for p in perms if not p.endswith(":*")}
    granted = set().union(*(role_perms.get(r, set()) for r in roles))
    return _expand(granted, universe) - _expand(denied, universe)

**Explanation**

`set().union(*iterables)` unions any number of sets in one call. Expanding wildcards against a "universe" set
keeps the algebra purely set-based: grant − deny.

**Complexity:** O(U · W) where U = universe size, W = wildcards.

**Common mistakes**

- Mutating `role_perms[r]` with `|=` — that changes the shared role definition for every user.
- Forgetting `set().union()` with zero args needs the `set()` start.

## Challenge — 🔴 Very Hard · Group duplicate customers (connected components)

**Working solution**

In [ ]:
def _phone(raw):
    digits = "".join(ch for ch in str(raw or "") if ch.isdigit())
    return digits[-10:] if len(digits) >= 10 else None

def duplicate_groups(customers):
    parent = {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]   # path halving
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    owner = {}                      # normalized key -> first customer id seen with it
    for c in customers:
        parent.setdefault(c["id"], c["id"])
        email = (c.get("email") or "").strip().lower() or None
        keys = [("email", email), ("phone", _phone(c.get("phone")))]
        for key in keys:
            if key[1] is None:
                continue
            if key in owner:
                union(owner[key], c["id"])
            else:
                owner[key] = c["id"]

    groups = {}
    for cid in parent:
        groups.setdefault(find(cid), []).append(cid)
    return sorted((sorted(g) for g in groups.values() if len(g) > 1), key=lambda g: g[0])

**Explanation**

Union-find (disjoint sets) turns "transitive matching" into near-linear time: every record is unioned with the first
record that owned each of its keys. This is the core of entity resolution / customer-360 dedupe jobs.

**Complexity:** ≈O(n · α(n)) — effectively linear.

**Common mistakes**

- Only grouping by email OR phone separately, missing transitive links.
- Pairwise comparisons: 20k records → 200M comparisons.